# WSI Preprocessing (WSG): Bag Indexing + Split Alignment

This notebook prepares Whole Slide Image (WSI) features for graph-based analysis.

We begin by:
1. Locating all pre-extracted WSI “bag” feature files (`.h5`) produced during WSI download/feature extraction.
2. Parsing TCGA identifiers from filenames to recover `slide_id` and `patient_id`.
3. Building an index table (`bags_df`) that links each bag file to its patient.

In later cells, we will:
- align bags to the saved train/val/test patient splits (7:1:2),
- perform quality filtering (e.g., minimum number of patches),
- and save the final split-aligned WSI feature sets for downstream graph construction.

In [1]:
import re
from pathlib import Path
import pandas as pd

BASE = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping")
BAGS_DIR = BASE / "Data" / "bags"

print("BAGS_DIR:", BAGS_DIR)
print("Exists?:", BAGS_DIR.exists())

bag_paths = sorted(BAGS_DIR.rglob("*.h5"))

print(f"\nTotal .h5 bag files found: {len(bag_paths)}")
if len(bag_paths) == 0:
    raise FileNotFoundError(f"No .h5 files found under: {BAGS_DIR}")

# Extract TCGA slide_id from filename (e.g., TCGA-4B-A93V.h5)
tcga_pattern = re.compile(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", re.IGNORECASE)

rows = []
for p in bag_paths:
    m = tcga_pattern.search(p.name)
    slide_id = m.group(1).upper() if m else None
    patient_id = slide_id[:12] if slide_id else None  # TCGA-XX-YYYY
    rows.append({
        "bag_path": str(p),
        "filename": p.name,
        "slide_id": slide_id,
        "patient_id": patient_id
    })

bags_df = pd.DataFrame(rows)

print("\nPreview (first 10):")
display(bags_df.head(10))

print("\nQC checks:")
print("Bags with missing slide_id:", bags_df["slide_id"].isna().sum())
print("Unique slide_ids:", bags_df["slide_id"].nunique(dropna=True))
print("Unique patient_ids:", bags_df["patient_id"].nunique(dropna=True))

print("\nExample slide_ids:")
print(bags_df["slide_id"].dropna().unique()[:10])

BAGS_DIR: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/bags
Exists?: True

Total .h5 bag files found: 830

Preview (first 10):


,bag_path,filename,slide_id,patient_id
0,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4384.h5,TCGA-05-4384,TCGA-05-4384
1,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4390.h5,TCGA-05-4390,TCGA-05-4390
2,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4396.h5,TCGA-05-4396,TCGA-05-4396
3,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4405.h5,TCGA-05-4405,TCGA-05-4405
4,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4410.h5,TCGA-05-4410,TCGA-05-4410
5,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4415.h5,TCGA-05-4415,TCGA-05-4415
6,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4417.h5,TCGA-05-4417,TCGA-05-4417
7,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4424.h5,TCGA-05-4424,TCGA-05-4424
8,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4425.h5,TCGA-05-4425,TCGA-05-4425
9,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4427.h5,TCGA-05-4427,TCGA-05-4427



QC checks:
Bags with missing slide_id: 0
Unique slide_ids: 830
Unique patient_ids: 830

Example slide_ids:
['TCGA-05-4384' 'TCGA-05-4390' 'TCGA-05-4396' 'TCGA-05-4405'
 'TCGA-05-4410' 'TCGA-05-4415' 'TCGA-05-4417' 'TCGA-05-4424'
 'TCGA-05-4425' 'TCGA-05-4427']


## 2. Load Train/Val/Test Patient Splits and Align WSI Bags

To ensure consistent multimodal modeling, we reuse the saved patient-level splits (train/val/test = 7:1:2).
In this cell, we:

- Load the split patient ID lists from disk.
- Filter the WSI bag index (`bags_df`) to keep only patients present in the splits.
- Report split coverage (how many split patients have a corresponding WSI bag).
- Save the split-aligned WSI bag index for reuse in downstream steps.

In [3]:
import numpy as np

SPLITS_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/splits")
if not SPLITS_DIR.exists():
    raise FileNotFoundError(f"Splits directory not found: {SPLITS_DIR}")

# Expecting these from your RNA split notebook
train_pids = np.load(SPLITS_DIR / "train_pids.npy", allow_pickle=True).astype(str)
val_pids   = np.load(SPLITS_DIR / "val_pids.npy", allow_pickle=True).astype(str)
test_pids  = np.load(SPLITS_DIR / "test_pids.npy", allow_pickle=True).astype(str)

split_set = set(train_pids) | set(val_pids) | set(test_pids)

print("Split sizes:", {"train": len(train_pids), "val": len(val_pids), "test": len(test_pids)})
print("Total unique patients in splits:", len(split_set))

# Filter bags to split patients
bags_df["patient_id"] = bags_df["patient_id"].astype(str)
bags_in_splits = bags_df[bags_df["patient_id"].isin(split_set)].copy()

print("\nWSI bags aligned to splits:", len(bags_in_splits), "/", len(bags_df))
print("Unique patients with WSI in splits:", bags_in_splits["patient_id"].nunique())

# Coverage report per split
def coverage(pids, df):
    have = set(df["patient_id"].unique())
    return len(set(pids) & have), len(pids)

cov_train = coverage(train_pids, bags_in_splits)
cov_val   = coverage(val_pids, bags_in_splits)
cov_test  = coverage(test_pids, bags_in_splits)

print("\nCoverage by split (patients with WSI / total patients):")
print(f"  train: {cov_train[0]} / {cov_train[1]}")
print(f"  val  : {cov_val[0]} / {cov_val[1]}")
print(f"  test : {cov_test[0]} / {cov_test[1]}")

# Save split-aligned bag index
OUT_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_index = OUT_DIR / "wsi_bags_index_aligned.tsv"
bags_in_splits.to_csv(out_index, sep="\t", index=False)
print("\nSaved split-aligned WSI bag index:", out_index)

display(bags_in_splits.head(10))

Split sizes: {'train': 581, 'val': 83, 'test': 167}
Total unique patients in splits: 831

WSI bags aligned to splits: 830 / 830
Unique patients with WSI in splits: 830

Coverage by split (patients with WSI / total patients):
  train: 581 / 581
  val  : 83 / 83
  test : 166 / 167

Saved split-aligned WSI bag index: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg/wsi_bags_index_aligned.tsv


,bag_path,filename,slide_id,patient_id
0,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4384.h5,TCGA-05-4384,TCGA-05-4384
1,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4390.h5,TCGA-05-4390,TCGA-05-4390
2,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4396.h5,TCGA-05-4396,TCGA-05-4396
3,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4405.h5,TCGA-05-4405,TCGA-05-4405
4,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4410.h5,TCGA-05-4410,TCGA-05-4410
5,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4415.h5,TCGA-05-4415,TCGA-05-4415
6,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4417.h5,TCGA-05-4417,TCGA-05-4417
7,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4424.h5,TCGA-05-4424,TCGA-05-4424
8,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4425.h5,TCGA-05-4425,TCGA-05-4425
9,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4427.h5,TCGA-05-4427,TCGA-05-4427


## 3. Inspect H5 Bag Structure and Compute Patch Counts (Quality Control)

Each WSI bag (`.h5`) stores patch-level feature embeddings extracted from a slide.
In this cell, we:

1. Inspect a few `.h5` files to confirm the dataset keys and feature matrix shape.
2. Identify the dataset key that contains patch embeddings (expected: `features`).
3. Iterate through all WSI bags and compute:
   - `num_patches`: number of patch embeddings in the bag
   - `embed_dim`: embedding dimension (expected: 1024 for UNI)
4. Handle unreadable/corrupted `.h5` files gracefully by skipping them and logging errors.
5. Save a table of corrupted files so they can be regenerated later if needed.

In [5]:
import h5py
import numpy as np
import pandas as pd

# 1) Helper: identify feature dataset key inside a bag file
def find_feature_key(h5_path):
    """Return the most likely dataset key for patch embeddings in an H5 bag."""
    with h5py.File(h5_path, "r") as f:
        keys = list(f.keys())

        # Common conventions (try these first)
        preferred = ["features", "embeddings", "feats", "x"]
        for k in preferred:
            if k in keys:
                obj = f[k]
                if hasattr(obj, "shape") and len(obj.shape) == 2:
                    return k

        # Otherwise: pick the first 2D dataset
        for k in keys:
            obj = f[k]
            if hasattr(obj, "shape") and len(obj.shape) == 2:
                return k

    return None

In [6]:
# 2) Inspect a few bags 
sample_paths = bags_in_splits["bag_path"].iloc[:5].tolist()

print("Inspecting first few bag files:")
for p in sample_paths:
    p = Path(p)
    try:
        key = find_feature_key(p)
        with h5py.File(p, "r") as f:
            print("\nFile:", p.name)
            print("Keys:", list(f.keys()))
            if key is None:
                print("No 2D feature matrix found.")
                continue
            arr = f[key]
            print("Feature key:", key, "| shape:", arr.shape, "| dtype:", arr.dtype)
    except Exception as e:
        print("\nFile:", p.name)
        print("Could not open/read:", repr(e))

Inspecting first few bag files:

File: TCGA-05-4384.h5
Keys: ['coords', 'features']
Feature key: features | shape: (79, 1024) | dtype: float32

File: TCGA-05-4390.h5
Keys: ['coords', 'features']
Feature key: features | shape: (770, 1024) | dtype: float32

File: TCGA-05-4396.h5
Keys: ['coords', 'features']
Feature key: features | shape: (2000, 1024) | dtype: float32

File: TCGA-05-4405.h5
Keys: ['coords', 'features']
Feature key: features | shape: (2000, 1024) | dtype: float32

File: TCGA-05-4410.h5
Keys: ['coords', 'features']
Feature key: features | shape: (454, 1024) | dtype: float32


In [7]:
# 3) Decide feature key (assume consistent across files)
FEATURE_KEY = find_feature_key(Path(bags_in_splits["bag_path"].iloc[0]))
if FEATURE_KEY is None:
    raise ValueError("Could not find a 2D feature dataset in the bag files.")

print("\nUsing FEATURE_KEY =", FEATURE_KEY)


Using FEATURE_KEY = features


In [8]:
# 4) Robust scan across all bags (skip corrupted)
num_patches = []
embed_dims = []
read_errors = []

for p in bags_in_splits["bag_path"].tolist():
    try:
        with h5py.File(p, "r") as f:
            if FEATURE_KEY not in f:
                raise KeyError(f"Missing dataset '{FEATURE_KEY}'")
            X = f[FEATURE_KEY]
            if len(X.shape) != 2:
                raise ValueError(f"Expected 2D features, got shape {X.shape}")

            num_patches.append(int(X.shape[0]))
            embed_dims.append(int(X.shape[1]))
            read_errors.append(None)

    except Exception as e:
        num_patches.append(np.nan)
        embed_dims.append(np.nan)
        read_errors.append(str(e))

bags_ok = bags_in_splits.copy()
bags_ok["num_patches"] = num_patches
bags_ok["embed_dim"] = embed_dims
bags_ok["read_error"] = read_errors

n_bad = bags_ok["read_error"].notna().sum()
print(f"\nTotal bags scanned: {len(bags_ok)}")
print(f"Readable bags: {len(bags_ok) - n_bad}")
print(f"Unreadable/corrupted bags: {n_bad}")

if n_bad > 0:
    print("\nExamples of unreadable bags:")
    display(bags_ok[bags_ok["read_error"].notna()][["bag_path", "filename", "read_error"]].head(10))

# Keep only readable bags for downstream QC
bags_ok = bags_ok[bags_ok["read_error"].isna()].copy()

print("\nEmbedding dim distribution (top):")
print(bags_ok["embed_dim"].value_counts().head())

print("\nPatch count summary (readable bags):")
print(bags_ok["num_patches"].describe())

display(bags_ok.sort_values("num_patches").head(10))

# 5) Save corrupted list 
OUT_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg")
OUT_DIR.mkdir(parents=True, exist_ok=True)

bad_out = OUT_DIR / "wsi_bad_h5_files.tsv"
(
    bags_in_splits.assign(read_error=read_errors)  # align length
    .loc[pd.Series(read_errors).notna().values, ["bag_path", "filename"]]
    .assign(read_error=pd.Series(read_errors)[pd.Series(read_errors).notna()].values)
    .to_csv(bad_out, sep="\t", index=False)
)
print("\nSaved corrupted/unreadable bag list to:", bad_out)


Total bags scanned: 830
Readable bags: 829
Unreadable/corrupted bags: 1

Examples of unreadable bags:


,bag_path,filename,read_error
466,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-75-6205.h5,Unable to synchronously open file (bad object ...



Embedding dim distribution (top):
embed_dim
1024.0    829
Name: count, dtype: int64

Patch count summary (readable bags):
count     829.000000
mean     1974.188179
std       195.239085
min        34.000000
25%      2000.000000
50%      2000.000000
75%      2000.000000
max      2000.000000
Name: num_patches, dtype: float64


,bag_path,filename,slide_id,patient_id,num_patches,embed_dim,read_error
12,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5425.h5,TCGA-05-5425,TCGA-05-5425,34.0,1024.0,None
8,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4425.h5,TCGA-05-4425,TCGA-05-4425,59.0,1024.0,None
0,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4384.h5,TCGA-05-4384,TCGA-05-4384,79.0,1024.0,None
15,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5715.h5,TCGA-05-5715,TCGA-05-5715,146.0,1024.0,None
11,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5423.h5,TCGA-05-5423,TCGA-05-5423,365.0,1024.0,None
289,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-55-8207.h5,TCGA-55-8207,TCGA-55-8207,450.0,1024.0,None
4,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4410.h5,TCGA-05-4410,TCGA-05-4410,454.0,1024.0,None
14,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5429.h5,TCGA-05-5429,TCGA-05-5429,498.0,1024.0,None
13,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5428.h5,TCGA-05-5428,TCGA-05-5428,591.0,1024.0,None
402,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-63-A5MH.h5,TCGA-63-A5MH,TCGA-63-A5MH,727.0,1024.0,None



Saved corrupted/unreadable bag list to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg/wsi_bad_h5_files.tsv


## 4. Apply Patch-Count QC and Save Final Split-Aligned WSI Index

Some slides contain very few tissue patches (e.g., due to poor tissue content or QC failures).
To reduce noise, we apply a minimum patch threshold (default: 100 patches).

In this cell, we:
- Filter readable WSI bags to keep only slides with `num_patches ≥ MIN_PATCHES`.
- Recompute train/val/test coverage after QC.
- Save the final QC-passed WSI bag index to disk for downstream WSI graph construction.

In [9]:
MIN_PATCHES = 100

bags_qc = bags_ok[bags_ok["num_patches"] >= MIN_PATCHES].copy()

print(f"QC threshold: num_patches >= {MIN_PATCHES}")
print("Slides kept:", len(bags_qc), "/", len(bags_ok))
print("Patients kept:", bags_qc["patient_id"].nunique(), "/", bags_ok["patient_id"].nunique())

# Coverage report after QC
have_patients = set(bags_qc["patient_id"].unique())

def cov(pids):
    return len(set(pids) & have_patients), len(pids)

cov_train = cov(train_pids)
cov_val   = cov(val_pids)
cov_test  = cov(test_pids)

print("\nCoverage AFTER QC (patients with WSI / total patients):")
print(f"  train: {cov_train[0]} / {cov_train[1]}")
print(f"  val  : {cov_val[0]} / {cov_val[1]}")
print(f"  test : {cov_test[0]} / {cov_test[1]}")

# Save final index
OUT_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_qc = OUT_DIR / f"wsi_bags_index_aligned_qc_min{MIN_PATCHES}.tsv"
bags_qc.to_csv(out_qc, sep="\t", index=False)
print("\nSaved QC-passed WSI bag index:", out_qc)

# Quick look at the lowest patch slides that passed QC
display(bags_qc.sort_values("num_patches").head(10))

QC threshold: num_patches >= 100
Slides kept: 826 / 829
Patients kept: 826 / 829

Coverage AFTER QC (patients with WSI / total patients):
  train: 579 / 581
  val  : 83 / 83
  test : 164 / 167

Saved QC-passed WSI bag index: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg/wsi_bags_index_aligned_qc_min100.tsv


,bag_path,filename,slide_id,patient_id,num_patches,embed_dim,read_error
15,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5715.h5,TCGA-05-5715,TCGA-05-5715,146.0,1024.0,None
11,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5423.h5,TCGA-05-5423,TCGA-05-5423,365.0,1024.0,None
289,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-55-8207.h5,TCGA-55-8207,TCGA-55-8207,450.0,1024.0,None
4,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4410.h5,TCGA-05-4410,TCGA-05-4410,454.0,1024.0,None
14,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5429.h5,TCGA-05-5429,TCGA-05-5429,498.0,1024.0,None
13,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-5428.h5,TCGA-05-5428,TCGA-05-5428,591.0,1024.0,None
402,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-63-A5MH.h5,TCGA-63-A5MH,TCGA-63-A5MH,727.0,1024.0,None
1,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-05-4390.h5,TCGA-05-4390,TCGA-05-4390,770.0,1024.0,None
195,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-50-5051.h5,TCGA-50-5051,TCGA-50-5051,1111.0,1024.0,None
720,/home/steps4growth/gmriechi/Lung-Cancer-Subtyp...,TCGA-97-8547.h5,TCGA-97-8547,TCGA-97-8547,1201.0,1024.0,None


## 5. Build Patient-Level WSI Embeddings (Mean Pooling) and Save Split Arrays

Graph construction and GNN training require one feature vector per patient.
Each WSI bag contains patch-level embeddings (N_patches × 1024). We convert each bag to a
single patient-level embedding by mean pooling across patches.

In this cell, we:
- Load each QC-passed bag and mean-pool patch embeddings → a 1024-d vector per patient.
- Construct split-specific matrices (train/val/test) using the saved patient splits.
- Save `X_wsi_{train,val,test}.npy`, `y_{train,val,test}.npy`, and patient id lists for consistent downstream modeling.

In [11]:
import numpy as np
import h5py
from tqdm import tqdm

FEATURE_KEY = "features"  # confirmed
EMB_DIM = 1024

# Map: patient_id -> bag_path (if multiple per patient ever existed, we'd choose one; here it's 1:1)
pid_to_bag = dict(zip(bags_qc["patient_id"].astype(str), bags_qc["bag_path"].astype(str)))

def mean_pool_bag(h5_path, key=FEATURE_KEY):
    with h5py.File(h5_path, "r") as f:
        X = f[key][()]  # loads into memory; ok since capped at 2000×1024
    # mean pooling over patches
    v = X.mean(axis=0).astype(np.float32)
    # optional L2 normalize (good for similarity graphs)
    norm = np.linalg.norm(v) + 1e-12
    v = v / norm
    return v

def build_split_arrays(split_pids, split_y, pid_to_bag):
    kept_pids = []
    X_list = []
    y_list = []

    for pid, y in zip(split_pids, split_y):
        pid = str(pid)
        if pid not in pid_to_bag:
            continue
        try:
            v = mean_pool_bag(pid_to_bag[pid])
        except Exception:
            # if anything goes wrong reading, skip
            continue
        kept_pids.append(pid)
        X_list.append(v)
        y_list.append(int(y))

    X = np.stack(X_list, axis=0).astype(np.float32) if len(X_list) else np.zeros((0, EMB_DIM), np.float32)
    y = np.array(y_list, dtype=np.int64)
    kept_pids = np.array(kept_pids, dtype=str)
    return X, y, kept_pids

# Load labels (same as RNA split notebook)
y_train = np.load(SPLITS_DIR / "y_train.npy").astype(np.int64)
y_val   = np.load(SPLITS_DIR / "y_val.npy").astype(np.int64)
y_test  = np.load(SPLITS_DIR / "y_test.npy").astype(np.int64)

X_train_wsi, y_train_wsi, pid_train_wsi = build_split_arrays(train_pids, y_train, pid_to_bag)
X_val_wsi,   y_val_wsi,   pid_val_wsi   = build_split_arrays(val_pids,   y_val,   pid_to_bag)
X_test_wsi,  y_test_wsi,  pid_test_wsi  = build_split_arrays(test_pids,  y_test,  pid_to_bag)

print("WSI arrays (after QC + pooling):")
print("Train:", X_train_wsi.shape, y_train_wsi.shape, "| patients:", len(pid_train_wsi))
print("Val  :", X_val_wsi.shape,   y_val_wsi.shape,   "| patients:", len(pid_val_wsi))
print("Test :", X_test_wsi.shape,  y_test_wsi.shape,  "| patients:", len(pid_test_wsi))

# Save outputs for downstream graph construction + modeling
OUT_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg")
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUT_DIR / "wsi_train_embeddings.npy", X_train_wsi)
np.save(OUT_DIR / "wsi_val_embeddings.npy",   X_val_wsi)
np.save(OUT_DIR / "wsi_test_embeddings.npy",  X_test_wsi)

np.save(OUT_DIR / "wsi_train_y.npy", y_train_wsi)
np.save(OUT_DIR / "wsi_val_y.npy",   y_val_wsi)
np.save(OUT_DIR / "wsi_test_y.npy",  y_test_wsi)

np.save(OUT_DIR / "wsi_train_patient_ids.npy", pid_train_wsi)
np.save(OUT_DIR / "wsi_val_patient_ids.npy",   pid_val_wsi)
np.save(OUT_DIR / "wsi_test_patient_ids.npy",  pid_test_wsi)

print("\nSaved pooled WSI embeddings + labels + patient IDs to:", OUT_DIR)

WSI arrays (after QC + pooling):
Train: (577, 1024) (577,) | patients: 577
Val  : (83, 1024) (83,) | patients: 83
Test : (164, 1024) (164,) | patients: 164

Saved pooled WSI embeddings + labels + patient IDs to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg


## 6. Construct the Patient-Level WSI Similarity Graph (WSG)

To build the Whole Slide Image Spatial Graph (WSG), we represent each patient as a node with a
1024-dimensional WSI embedding (mean-pooled UNI features).

Edges are constructed by connecting each node to its k nearest neighbors under cosine similarity,
yielding a sparse patient–patient graph suitable for GAT-based learning.

In this cell, we:
- Build kNN graphs separately for train/val/test (or train-only for training, depending on modeling choice).
- Produce `edge_index` in PyTorch Geometric format.
- Save graph artifacts (edge_index, similarities) for downstream GAT modeling.

In [12]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

OUT_DIR = Path("/home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def build_knn_edge_index(X, k=10, metric="cosine", self_loops=False):
    """
    Build PyG-style edge_index (2, E) using kNN under cosine distance.
    Returns:
      edge_index (np.int64), edge_weight (np.float32)
    """
    n = X.shape[0]
    if n == 0:
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    # k+1 because nearest neighbor is itself (distance=0) for cosine
    nn = NearestNeighbors(n_neighbors=min(k + 1, n), metric=metric, algorithm="auto")
    nn.fit(X)
    dists, idxs = nn.kneighbors(X, return_distance=True)

    src = []
    dst = []
    wts = []

    for i in range(n):
        neigh = idxs[i]
        dist  = dists[i]
        for j, d in zip(neigh, dist):
            if (not self_loops) and (j == i):
                continue
            src.append(i)
            dst.append(int(j))
            # cosine similarity = 1 - cosine distance
            wts.append(float(1.0 - d))

    edge_index = np.vstack([np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)])
    edge_weight = np.array(wts, dtype=np.float32)
    return edge_index, edge_weight

K = 10  # you can later sweep K={5,10,15} like in your other graphs

edge_train, w_train = build_knn_edge_index(X_train_wsi, k=K, metric="cosine", self_loops=False)
edge_val,   w_val   = build_knn_edge_index(X_val_wsi,   k=K, metric="cosine", self_loops=False)
edge_test,  w_test  = build_knn_edge_index(X_test_wsi,  k=K, metric="cosine", self_loops=False)

print("WSG edges:")
print("Train edge_index:", edge_train.shape, "| edge_weight:", w_train.shape)
print("Val   edge_index:", edge_val.shape,   "| edge_weight:", w_val.shape)
print("Test  edge_index:", edge_test.shape,  "| edge_weight:", w_test.shape)

# Save graph artifacts
np.save(OUT_DIR / f"wsg_train_edge_index_k{K}.npy", edge_train)
np.save(OUT_DIR / f"wsg_train_edge_weight_k{K}.npy", w_train)

np.save(OUT_DIR / f"wsg_val_edge_index_k{K}.npy", edge_val)
np.save(OUT_DIR / f"wsg_val_edge_weight_k{K}.npy", w_val)

np.save(OUT_DIR / f"wsg_test_edge_index_k{K}.npy", edge_test)
np.save(OUT_DIR / f"wsg_test_edge_weight_k{K}.npy", w_test)

print("\nSaved WSG graph artifacts to:", OUT_DIR)

WSG edges:
Train edge_index: (2, 5770) | edge_weight: (5770,)
Val   edge_index: (2, 830) | edge_weight: (830,)
Test  edge_index: (2, 1640) | edge_weight: (1640,)

Saved WSG graph artifacts to: /home/steps4growth/gmriechi/Lung-Cancer-Subtyping/Data/02_preprocessed/wsi_wsg
